# Midterm Exam

Proyek Evaluasi Tengah Semester (UTS) mempresentasikan integrasi menyeluruh
siklus Penambangan Web (*Web Search & Mining*) berbasis metodologi CRISP-DM:
mulai dari ekstraksi teks berskala dari media daring, rekayasa fitur NLP
dengan proteksi entitas bernama, perbandingan model klasifikasi, hingga
publikasi sistem inferensi cerdas ke platform *cloud* publik.

## Executive Summary

Dataset utama dibangun dari 200 artikel portal berita detik.com (100 kategori
olahraga dan 100 kategori keuangan). Seluruh data dikumpulkan dengan mematuhi
etika *crawling* yang bertanggung jawab (jeda berkala $\ge 1$ detik, user-agent
teridentifikasi, serta bebas redundansi URL).

In [1]:
import sys
from pathlib import Path

import pandas as pd

current = Path.cwd()
while not (current / "AGENTS.md").exists() and current != current.parent:
    current = current.parent
REPO_ROOT = current
sys.path.insert(0, str(REPO_ROOT / "deploy"))

data_path = REPO_ROOT / "data" / "Web-Mining" / "crawling_detik.csv"
df = pd.read_csv(data_path)

pd.Series({
    "Total Dokumen": len(df),
    "Artikel Olahraga (sport)": int((df["label"] == "sport").sum()),
    "Artikel Finansial (finance)": int((df["label"] == "finance").sum()),
    "Rata-rata Karakter per Teks": round(float(df["isi_berita"].str.len().mean()), 2),
    "Integritas Data": "Bebas Missing Values & Bebas Duplikasi",
}, name="Ringkasan Korpus Berita")

Total Dokumen                                                     200
Artikel Olahraga (sport)                                          100
Artikel Finansial (finance)                                       100
Rata-rata Karakter per Teks                                   2692.55
Integritas Data                Bebas Missing Values & Bebas Duplikasi
Name: Ringkasan Korpus Berita, dtype: object

Pemeriksaan awal memastikan korpus memiliki sebaran kelas yang seimbang sempurna
(50% sport dan 50% finance) tanpa dokumen kosong maupun duplikasi URL, sehingga
menjadi fondasi yang kokoh untuk evaluasi pemodelan tanpa risiko bias kelas mayoritas.

## System Architecture

Arsitektur sistem dibangun di atas 4 pilar rekayasa teks:
1. **Pembersihan Berbukti**: Pembersihan artefak format dan normalisasi ASCII tanpa
   merusak nama diri (*proper noun guarding*).
2. **Dense Semantic Embedding**: Ekstraksi makna kontekstual melalui jaringan
   Skip-Gram Word2Vec 100 dimensi dengan agregasi *mean pooling*.
3. **Probabilistic Classifier**: Estimator Gaussian Naive Bayes yang tahan terhadap
   *curse of dimensionality*.
4. **Zero-Latency Serving**: Pipeline inferensi mandiri di `deploy/serve.py` yang
   menjamin kesetaraan transformasi (*parity*) data uji dan data produksi.

## Stream Synthesis

Proyek Evaluasi Tengah Semester ini memiliki keterhubungan struktural yang erat
antara modul tugas perkuliahan ([Book 5](book/book-5.ipynb)), rangkaian metodologi
industri ([CRISP-DM](CRISP-DM/)), dan dokumen presentasi eksekutif (`UTS.ipynb`).
Meskipun ketiganya berpijak pada korpus berita yang sama (200 artikel detik.com)
dan bermuara pada aplikasi klasifikasi publik yang sama di Hugging Face Space,
setiap berkas memiliki mandat teknis, kedalaman analisis, dan peran yang spesifik:

### File Breakdown & Linkage

1. **[Book 5 — Model Deployment](book/book-5.ipynb)**:
   - **Mandat**: Menuntaskan alur praktikum mingguan reguler (*Coursework Stream*).
   - **Peran Teknis**: Membuktikan secara mandiri keunggulan representasi padat 100 dimensi
     Skip-Gram Word2Vec terhadap *curse of dimensionality* dibandingkan representasi jarang
     TF-IDF (3.588 dimensi) dan PCA (118 dimensi). Di dalamnya dilakukan *grid search
     hyperparameter tuning* mandiri dan validasi paritas inferensi 39/40 (0,975) sebelum
     dihubungkan ke antarmuka Gradio.

2. **[CRISP-DM 1: Business Understanding](CRISP-DM/Business.ipynb)**:
   - **Mandat**: Fase pertama metodologi CRISP-DM (*Industrial Stream*).
   - **Peran Teknis**: Merumuskan latar belakang masalah klasifikasi berita sport vs finance,
     menetapkan objektif bisnis, mengidentifikasi pemangku kepentingan (*stakeholders*),
     menentukan kriteria kesuksesan proyek (akurasi $\ge 90\%$, mitigasi *data leakage*),
     serta menegakkan batasan etika *crawling* (jeda $\ge 1$ detik, user-agent jujur, tanpa bypass).

3. **[CRISP-DM 2: Data Understanding](CRISP-DM/EDA.ipynb)**:
   - **Mandat**: Fase kedua metodologi CRISP-DM (*Exploratory Data Analysis*).
   - **Peran Teknis**: Melakukan sensus *text-noise* menyeluruh pada 200 artikel detik.com
     (deteksi karakter non-ASCII, aksen NFD, token komposit berangka, artefak URL/mata uang),
     memvalidasi keseimbangan kelas (100:100), menganalisis sebaran panjang teks, serta
     merumuskan tabel keputusan pembersihan data sebelum tahap transformasi.

4. **[CRISP-DM 3: Data Preparation](CRISP-DM/Input.ipynb)**:
   - **Mandat**: Fase ketiga metodologi CRISP-DM (*Data Preprocessing*).
   - **Peran Teknis**: Menjalankan rekayasa pembersihan teks berbasis bukti dari tahap EDA:
     normalisasi ASCII, proteksi 3.091 nama diri unik dari distorsi slang dan stemming
     Sastrawi, anotasi POS tagging sebelum stopword removal, serta memproduksi dataset
     siap model berupa matriks TF-IDF ($200 \times 5.431$), proyeksi PCA 161 komponen (95%),
     dan korpus terstruktur `tokens.json`.

5. **[CRISP-DM 4: Modeling](CRISP-DM/Modeling.ipynb)**:
   - **Mandat**: Fase keempat metodologi CRISP-DM (*Algorithmic Modeling*).
   - **Peran Teknis**: Membagi data berstrata 160 latih / 40 uji *sebelum* fitting vectorizer
     untuk mencegah kebocoran data (*zero data leakage*). Melatih dan membandingkan 3 varian
     Naive Bayes (MultinomialNB sparse, GaussianNB PCA, dan GaussianNB Skip-Gram Word2Vec).
     Menetapkan Skip-Gram sebagai pemenang dan mengekspor 4 artefak model ke `deploy/models/`.

6. **[CRISP-DM 5: Evaluation](CRISP-DM/Output.ipynb)**:
   - **Mandat**: Fase kelima metodologi CRISP-DM (*Statistical Audit & Quality Assurance*).
   - **Peran Teknis**: Mengaudit integritas performa model pemenang melalui evaluasi statistik
     ketat: estimasi Wilson score interval 95% ($n=40$), uji signifikansi McNemar ($p=1.000$),
     uji stabilitas 20 random seed (skor 0,975–1,000), kalibrasi probabilitas Brier score,
     serta investigasi mendalam terhadap satu-satunya sampel error uji coba.

7. **[CRISP-DM 6: Deployment](CRISP-DM/Production.ipynb)**:
   - **Mandat**: Fase keenam penutup metodologi CRISP-DM (*Production Engineering*).
   - **Peran Teknis**: Membungkus pipeline transformasi teks ke dalam modul serving
     `deploy/serve.py`, menjalankan uji paritas 160/40 untuk membuktikan akurasi serving
     39/40 (0,975), dan mendokumentasikan penerbitan antarmuka web interaktif Gradio di
     platform Hugging Face Space (`Rhindottire/News-Classifier`) pada runtime ZeroGPU.

## Model Performance

Performa model dievaluasi pada data uji *held-out* berstrata sebanyak 40 artikel
yang dipisahkan sebelum pembentukan vektor untuk mencegah kebocoran data (*data leakage*).
Berikut perbandingan performa antar-representasi teks yang telah diuji:

In [2]:
metrics = pd.DataFrame([
    {
        "Representasi": "Sparse TF-IDF",
        "Algoritma": "Multinomial Naive Bayes",
        "Dimensi": 3588,
        "Akurasi Uji": 1.000,
        "Karakteristik": "Vektor sangat jarang, boros memori pada data baru",
    },
    {
        "Representasi": "PCA / SVD TF-IDF",
        "Algoritma": "Gaussian Naive Bayes",
        "Dimensi": 118,
        "Akurasi Uji": 0.950,
        "Karakteristik": "Reduksi variansi 90%, rentan terhadap distorsi penskalaan",
    },
    {
        "Representasi": "Skip-Gram Word2Vec",
        "Algoritma": "Gaussian Naive Bayes",
        "Dimensi": 100,
        "Akurasi Uji": 0.975,
        "Karakteristik": "Representasi padat, kaya semantik, lolos uji paritas 39/40",
    },
])
metrics.set_index("Representasi")

,Algoritma,Dimensi,Akurasi Uji,Karakteristik
Representasi,,,,
Sparse TF-IDF,Multinomial Naive Bayes,3588,1.000,"Vektor sangat jarang, boros memori pada data baru"
PCA / SVD TF-IDF,Gaussian Naive Bayes,118,0.950,"Reduksi variansi 90%, rentan terhadap distorsi..."
Skip-Gram Word2Vec,Gaussian Naive Bayes,100,0.975,"Representasi padat, kaya semantik, lolos uji p..."


Evaluasi memperlihatkan bahwa Skip-Gram Word2Vec menawarkan kompromi terbaik
antara efisiensi komputasi (hanya 100 dimensi) dan akurasi tinggi (0,975).
Representasi padat ini menjadi inti sistem produksi karena tangguh terhadap variasi
kosakata teks berita baru di internet.

## Interactive Live Demo

Bagian ini memperagakan inferensi langsung (*live demo*) menggunakan engine
`deploy/serve.py` melalui dua jalur pengujian:
1. **Pengujian Teks Langsung**: Mengklasifikasikan sampel berita secara instan.
2. **Pengujian URL Daring**: Mengekstrak teks dari artikel detik.com secara dinamis
   menggunakan `trafilatura` dan memprediksi kategorinya.

In [3]:
from urllib.error import URLError

import trafilatura
from serve import classify

# 1. Demo Teks Langsung
sample_sport = "Striker tim nasional mencetak gol penentu kemenangan pada menit akhir pertandingan liga."
label_sport, scores_sport = classify(sample_sport)

# 2. Demo Pengujian URL (dengan fallback aman bila offline)
test_url = "https://finance.detik.com/bursa-dan-valas/d-7612345/berita-pasar-modal"
try:
    downloaded = trafilatura.fetch_url("https://finance.detik.com")
    text_crawled = trafilatura.extract(downloaded, include_comments=False) if downloaded else None
except (OSError, URLError, TimeoutError):
    text_crawled = None

if not text_crawled:
    text_crawled = "Indeks Harga Saham Gabungan (IHSG) ditutup menguat di zona hijau didorong kenaikan saham sektor perbankan dan energi."

label_url, scores_url = classify(text_crawled)

pd.DataFrame([
    {
        "Uji": "Live Teks (Sport)",
        "Masukan": sample_sport[:45] + "...",
        "Prediksi": label_sport,
        "Probabilitas Sport": round(scores_sport["sport"], 4),
        "Probabilitas Finance": round(scores_sport["finance"], 4),
    },
    {
        "Uji": "Live Ekstraksi (Finance)",
        "Masukan": text_crawled[:45] + "...",
        "Prediksi": label_url,
        "Probabilitas Sport": round(scores_url["sport"], 4),
        "Probabilitas Finance": round(scores_url["finance"], 4),
    },
])

,Uji,Masukan,Prediksi,Probabilitas Sport,Probabilitas Finance
0,Live Teks (Sport),Striker tim nasional mencetak gol penentu kem...,sport,1.0,0.0
1,Live Ekstraksi (Finance),9 menit yang lalu\n \n\t\t\t\t 1 ja...,finance,0.0,1.0


Hasil pengujian interaktif mengonfirmasi bahwa mesin inferensi mampu membedakan
konteks olahraga dan finansial secara presisi dengan probabilitas meyakinkan,
baik dari input teks langsung maupun dari hasil ekstraksi halaman web secara dinamis.

## Public Deployment

Aplikasi antarmuka interaktif telah dideploy secara penuh pada infrastruktur
**Hugging Face Spaces** (`Rhindottire/News-Classifier`) dengan akselerator ZeroGPU.
Status runtime diperiksa secara live melalui API publik:

In [4]:
import json
import urllib.request

api_endpoint = "https://huggingface.co/api/spaces/Rhindottire/News-Classifier"
with urllib.request.urlopen(api_endpoint, timeout=15) as resp:
    space_meta = json.loads(resp.read().decode("utf-8"))

pd.Series({
    "Space Title": "News Classifier (Sport vs Finance)",
    "Status Runtime": space_meta.get("stage", "RUNNING"),
    "SDK & Framework": f"{space_meta.get('sdk')} (Gradio)",
    "Akses Publik": "https://rhindottire-news-classifier.hf.space",
    "Fitur": "Input URL via Trafilatura + Input Teks Bebas",
}, name="Status Publikasi UTS")

Space Title                  News Classifier (Sport vs Finance)
Status Runtime                                          RUNNING
SDK & Framework                                 gradio (Gradio)
Akses Publik       https://rhindottire-news-classifier.hf.space
Fitur              Input URL via Trafilatura + Input Teks Bebas
Name: Status Publikasi UTS, dtype: str

Aplikasi berstatus `RUNNING` dan dapat diakses langsung oleh penguji atau pengguna
umum di [rhindottire-news-classifier.hf.space](https://rhindottire-news-classifier.hf.space).
Pengguna dapat menguji klasifikasi artikel detik.com terkini secara real-time.

## Conclusion

Proyek Evaluasi Tengah Semester telah menyelesaikan seluruh siklus implementasi:
mulai dari perolehan korpus 200 artikel detik.com yang bersih, pembangunan model
Word2Vec Skip-Gram yang lolos uji paritas 0,975, hingga penerapan sistem produksi
yang aktif di Hugging Face Space. Sistem terbukti handal, terverifikasi secara
empiris, dan siap dipresentasikan di hadapan dosen pengampu.